In [0]:
#-------------------------------------------------------------------------------------------#
# Data criacao: 15/11/2025                                                                  #
# Objetivo:                                                                                 #
#   Criar uma tabela de Clientes para equipe de marketing:                                  #   #   Importante somente pessoas que tenham e-mail e tefefone cadastrados                     #  
#    Origem camada bronze: customers                                                        #
#    Destino camada silver: customers                                                       #
#-------------------------------------------------------------------------------------------#
# 1. Criar uma tabela temporária com os dados da tabela bronze                              #
# 2. Criar uma tabela Silver com os dados da tabela bronze                                  #
# 4. Importar as funções desejadas                                                          #
#-------------------------------------------------------------------------------------------#

In [0]:
import sys
import importlib

# 1. Garante que alterações no arquivo .py sejam carregadas automaticamente
%load_ext autoreload
%autoreload 2

# 2. Adiciona a pasta pai ('Codigo') com o prefixo '/Workspace'
import os; caminho_codigo = os.path.abspath("../..")
if caminho_codigo not in sys.path:
    sys.path.append(caminho_codigo)

# 3. Importa e força o recarregamento do módulo
import funcoes_diversas.funcoes_util
importlib.reload(funcoes_diversas.funcoes_util)

# 4. Importa as funções desejadas
from funcoes_diversas.funcoes_util import buscar_caminho, gerar_tabela_temporaria



The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [0]:
mapa_criado = gerar_tabela_temporaria(["customers"])

View 'tmp_bronze_customers' criada com sucesso!


In [0]:
df_customers_silver = spark.sql("""
    select C.customer_id,
           C.first_name,
           C.last_name,
           C.phone,
           C.email,
           C.street,
           C.city,
           C.state,
           C.zip_code           
    from tmp_bronze_customers as C
    where 1 = 1
      and C.email is not null -- vazio
      and C.email not in('NULL', 'NULL ') -- com palavra null
      and C.phone is not null -- vazio
      and C.phone not in('NULL', 'NULL ') -- com palavra null
""")


customer_id,first_name,last_name,phone,email,street,city,state,zip_code
5,Charolette,Rice,(916) 381-6003,charolette.rice@msn.com,107 River Dr.,Sacramento,CA,95820
7,Latasha,Hays,(716) 986-3359,latasha.hays@hotmail.com,7014 Manor Station Rd.,Buffalo,NY,14215
12,Robby,Sykes,(516) 583-7761,robby.sykes@hotmail.com,486 Rock Maple Street,Hempstead,NY,11550
16,Emmitt,Sanchez,(212) 945-8823,emmitt.sanchez@hotmail.com,461 Squaw Creek Road,New York,NY,10002
26,Theo,Reese,(562) 215-2907,theo.reese@gmail.com,8755 W. Wild Horse St.,Long Beach,NY,11561
31,Williemae,Holloway,(510) 246-8375,williemae.holloway@msn.com,69 Cypress St.,Oakland,CA,94603
39,Janetta,Aguirre,(717) 670-2634,janetta.aguirre@aol.com,214 Second Court,Lancaster,NY,14086
43,Mozelle,Carter,(281) 489-9656,mozelle.carter@aol.com,895 Chestnut Ave.,Houston,TX,77016
56,Lolita,Mosley,(281) 363-3309,lolita.mosley@hotmail.com,376 S. High Ridge St.,Houston,TX,77016
67,Tommie,Melton,(916) 802-2952,tommie.melton@gmail.com,8 West Old York St.,Sacramento,CA,95820


In [0]:
silver_path = buscar_caminho("silver")
# transformando consulta em uma tabela Silver 
df_customers_silver.write\
    .mode("overwrite")\
    .format("delta")\
    .option("overwriteSchema", "true")\
    .save(f"{silver_path}/customers")